# SQL Analysis — NYC Yellow Taxi

## Objetivo

Analizar los datos de Yellow Taxi de julio de 2026 mediante SQL, utilizando DuckDB para consultar directamente los archivos Parquet.

El análisis se enfocará en métricas de demanda, ingresos, movilidad y distribución geográfica.

In [1]:
# Preparar el entorno SQL
from pathlib import Path
import pandas as pd
import duckdb

# Evitar notación científica
pd.options.display.float_format = "{:,.2f}".format

# Crear conexión con DuckDB
con = duckdb.connect()

print("DuckDB conectado correctamente")

DuckDB conectado correctamente


In [2]:
# Definir la ruta del dataset preparado
ruta_features = "../data/processed/yellow_tripdata_2026-07_features.parquet"

# Verificar el número total de registros del dataset
consulta_total = f"""
SELECT
    COUNT(*) AS total_viajes
FROM read_parquet('{ruta_features}')
"""

con.execute(consulta_total).df()

,total_viajes
0,3530109


In [3]:
# Calcular KPIs generales con SQL
consulta_kpis = f"""
SELECT
    COUNT(*) AS total_viajes,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio,
    ROUND(AVG(trip_duration_min), 2) AS duracion_promedio,
    ROUND(AVG(total_amount), 2) AS importe_promedio
FROM read_parquet('{ruta_features}')
WHERE flag_quality_issue = FALSE
"""

con.execute(consulta_kpis).df()

,total_viajes,distancia_promedio,duracion_promedio,importe_promedio
0,3359619,3.57,17.24,29.89


In [4]:
# Analizar la cantidad de viajes por día de la semana con SQL
consulta_dias = f"""
SELECT
    pickup_day AS dia_semana,
    COUNT(*) AS cantidad_viajes
FROM read_parquet('{ruta_features}')
WHERE flag_quality_issue = FALSE
GROUP BY pickup_day
ORDER BY cantidad_viajes DESC
"""

con.execute(consulta_dias).df()

,dia_semana,cantidad_viajes
0,Thursday,574632
1,Wednesday,569790
2,Friday,529043
3,Saturday,448676
4,Tuesday,428308
5,Sunday,416606
6,Monday,392564


In [5]:
# Analizar la demanda de viajes por hora con SQL
consulta_horas = f"""
SELECT
    pickup_hour AS hora,
    COUNT(*) AS cantidad_viajes
FROM read_parquet('{ruta_features}')
WHERE flag_quality_issue = FALSE
GROUP BY pickup_hour
ORDER BY cantidad_viajes DESC
"""

con.execute(consulta_horas).df()

,hora,cantidad_viajes
0,18,210924
1,17,205614
2,21,203527
3,14,195593
4,15,195339
5,19,194593
6,20,194512
7,22,193848
8,16,188138
9,13,183478


In [6]:
# Analizar viajes e ingresos por hora
consulta_ingresos_hora = f"""
SELECT
    pickup_hour AS hora,
    COUNT(*) AS cantidad_viajes,
    ROUND(SUM(total_amount), 2) AS ingresos_totales,
    ROUND(AVG(total_amount), 2) AS importe_promedio
FROM read_parquet('{ruta_features}')
WHERE
    flag_quality_issue = FALSE
    AND total_amount > 0
GROUP BY pickup_hour
ORDER BY ingresos_totales DESC
"""

con.execute(consulta_ingresos_hora).df()

,hora,cantidad_viajes,ingresos_totales,importe_promedio
0,17,204709,"6,383,944.60",31.19
1,18,209993,"6,237,368.31",29.70
2,14,194765,"6,093,657.24",31.29
3,15,194550,"6,079,175.79",31.25
4,16,187243,"6,076,651.83",32.45
5,21,202786,"5,945,911.00",29.32
6,22,193147,"5,837,889.87",30.23
7,19,193707,"5,737,006.94",29.62
8,20,193736,"5,634,001.33",29.08
9,13,182750,"5,456,172.85",29.86


In [7]:
# Analizar viajes e ingresos por día de la semana
consulta_ingresos_dia = f"""
SELECT
    pickup_day AS dia_semana,
    COUNT(*) AS cantidad_viajes,
    ROUND(SUM(total_amount), 2) AS ingresos_totales,
    ROUND(AVG(total_amount), 2) AS importe_promedio
FROM read_parquet('{ruta_features}')
WHERE
    flag_quality_issue = FALSE
    AND total_amount > 0
GROUP BY pickup_day
ORDER BY ingresos_totales DESC
"""

con.execute(consulta_ingresos_dia).df()

,dia_semana,cantidad_viajes,ingresos_totales,importe_promedio
0,Thursday,572385,"17,341,393.49",30.30
1,Wednesday,567514,"17,043,369.02",30.03
2,Friday,526749,"15,800,585.68",30.00
3,Saturday,446891,"13,041,842.12",29.18
4,Sunday,415117,"12,821,843.23",30.89
5,Tuesday,426620,"12,673,544.26",29.71
6,Monday,390988,"12,101,939.46",30.95


In [8]:
# Analizar los viajes según categoría de distancia
consulta_distancia = f"""
SELECT
    distance_category AS categoria_distancia,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio,
    ROUND(AVG(trip_duration_min), 2) AS duracion_promedio,

    ROUND(AVG(total_amount), 2) AS importe_promedio
FROM read_parquet('{ruta_features}')
WHERE flag_quality_issue = FALSE
GROUP BY distance_category
ORDER BY distancia_promedio
"""

con.execute(consulta_distancia).df()

,categoria_distancia,cantidad_viajes,distancia_promedio,duracion_promedio,importe_promedio
0,Short,1694587,1.13,10.00,18.73
1,Medium,986137,3.12,19.08,28.96
2,Long,425852,7.11,26.56,44.91
3,Very Long,253043,15.76,42.88,82.99


In [9]:
# Mostrar el nombre del tipo de pago en lugar del código
consulta_propinas = f"""
SELECT
    CASE payment_type
        WHEN 0 THEN 'Flex Fare'
        WHEN 1 THEN 'Credit Card'
        WHEN 2 THEN 'Cash'
        WHEN 3 THEN 'No Charge'
        WHEN 4 THEN 'Dispute'
        WHEN 5 THEN 'Unknown'
        WHEN 6 THEN 'Voided Trip'
        ELSE 'Other'
    END AS tipo_pago,

    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(tip_amount), 2) AS propina_promedio,
    ROUND(AVG(tip_percentage), 2) AS porcentaje_propina_promedio

FROM read_parquet('{ruta_features}')

WHERE
    flag_quality_issue = FALSE
    AND total_amount > 0
    AND tip_amount >= 0
    AND tip_percentage BETWEEN 0 AND 100

GROUP BY payment_type
ORDER BY cantidad_viajes DESC
"""

con.execute(consulta_propinas).df()

,tipo_pago,cantidad_viajes,propina_promedio,porcentaje_propina_promedio
0,Credit Card,2121538,4.34,17.17
1,Flex Fare,877997,0.32,1.10
2,Cash,327941,0.00,0.00
3,Dispute,10776,0.00,0.01
4,No Charge,6328,0.00,0.00


In [10]:
# Analizar las principales zonas de origen con JOIN y HAVING
consulta_zonas = f"""
SELECT
    z.Zone AS zona,
    z.Borough AS borough,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(t.total_amount), 2) AS importe_promedio

FROM read_parquet('{ruta_features}') AS t

LEFT JOIN read_csv_auto('../data/raw/taxi_zone_lookup.csv') AS z
    ON t.PULocationID = z.LocationID

WHERE t.flag_quality_issue = FALSE

GROUP BY z.Zone, z.Borough

HAVING COUNT(*) > 50000

ORDER BY cantidad_viajes DESC
"""

con.execute(consulta_zonas).df()

,zona,borough,cantidad_viajes,importe_promedio
0,JFK Airport,Queens,147333,76.87
1,Midtown Center,Manhattan,147261,26.82
2,Upper East Side South,Manhattan,134923,21.69
3,Penn Station/Madison Sq West,Manhattan,113769,26.82
4,Upper East Side North,Manhattan,112912,22.48
5,Midtown East,Manhattan,108993,25.74
6,Times Sq/Theatre District,Manhattan,104945,30.14
7,Lincoln Square East,Manhattan,93653,23.54
8,Murray Hill,Manhattan,92972,25.32
9,East Chelsea,Manhattan,91942,27.67


In [11]:
# Analizar las zonas de origen que generan mayores ingresos
consulta_ingresos_zona = f"""
SELECT
    z.Zone AS zona,
    z.Borough AS borough,
    COUNT(*) AS cantidad_viajes,
    ROUND(SUM(t.total_amount), 2) AS ingresos_totales,
    ROUND(AVG(t.total_amount), 2) AS importe_promedio

FROM read_parquet('{ruta_features}') AS t

LEFT JOIN read_csv_auto('../data/raw/taxi_zone_lookup.csv') AS z
    ON t.PULocationID = z.LocationID

WHERE
    t.flag_quality_issue = FALSE
    AND t.total_amount > 0

GROUP BY z.Zone, z.Borough

ORDER BY ingresos_totales DESC

LIMIT 10
"""

con.execute(consulta_ingresos_zona).df()

,zona,borough,cantidad_viajes,ingresos_totales,importe_promedio
0,JFK Airport,Queens,145589,"11,444,479.49",78.61
1,LaGuardia Airport,Queens,86131,"6,000,936.37",69.67
2,Midtown Center,Manhattan,146682,"3,962,906.90",27.02
3,Times Sq/Theatre District,Manhattan,104314,"3,179,746.03",30.48
4,Penn Station/Madison Sq West,Manhattan,113309,"3,062,436.76",27.03
5,Upper East Side South,Manhattan,134423,"2,934,749.55",21.83
6,Midtown East,Manhattan,108574,"2,815,266.12",25.93
7,East Chelsea,Manhattan,91595,"2,552,312.73",27.87
8,Upper East Side North,Manhattan,112607,"2,543,382.25",22.59
9,Murray Hill,Manhattan,92642,"2,360,522.36",25.48


In [12]:
# Crear una vista SQL para reutilizar el dataset
crear_vista = f"""
CREATE OR REPLACE VIEW trips AS
SELECT *
FROM read_parquet('{ruta_features}')
"""

con.execute(crear_vista)

print("Vista 'trips' creada correctamente")

Vista 'trips' creada correctamente


In [13]:
# Crear una vista con los registros válidos para análisis económico
crear_vista_economica = """
CREATE OR REPLACE VIEW trips_economic AS
SELECT *
FROM trips
WHERE
    flag_quality_issue = FALSE
    AND total_amount > 0
    AND tip_amount >= 0
    AND tip_percentage BETWEEN 0 AND 100
"""

con.execute(crear_vista_economica)

print("Vista 'trips_economic' creada correctamente")

Vista 'trips_economic' creada correctamente


In [14]:
# Comparar media y mediana de las principales métricas económicas
consulta_media_mediana = """
SELECT
    ROUND(AVG(total_amount), 2) AS importe_medio,
    ROUND(MEDIAN(total_amount), 2) AS importe_mediano,
    ROUND(AVG(tip_amount), 2) AS propina_media,
    ROUND(MEDIAN(tip_amount), 2) AS propina_mediana
FROM trips_economic
"""

con.execute(consulta_media_mediana).df()

,importe_medio,importe_mediano,propina_media,propina_mediana
0,30.12,23.57,2.83,2.00


In [15]:
# Calcular cuartiles e IQR del importe total con SQL
consulta_cuartiles = """
SELECT
    ROUND(QUANTILE_CONT(total_amount, 0.25), 2) AS Q1,
    ROUND(MEDIAN(total_amount), 2) AS mediana,
    ROUND(QUANTILE_CONT(total_amount, 0.75), 2) AS Q3,
    ROUND(
        QUANTILE_CONT(total_amount, 0.75)
        - QUANTILE_CONT(total_amount, 0.25),
        2
    ) AS IQR
FROM trips_economic
"""

con.execute(consulta_cuartiles).df()

,Q1,mediana,Q3,IQR
0,17.45,23.57,33.95,16.50


In [16]:
# Calcular la participación porcentual de cada categoría de distancia
consulta_participacion = """
SELECT
    distance_category AS categoria_distancia,
    COUNT(*) AS cantidad_viajes,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS participacion_pct
FROM trips
WHERE flag_quality_issue = FALSE
GROUP BY distance_category
ORDER BY cantidad_viajes DESC
"""

con.execute(consulta_participacion).df()

,categoria_distancia,cantidad_viajes,participacion_pct
0,Short,1694587,50.44
1,Medium,986137,29.35
2,Long,425852,12.68
3,Very Long,253043,7.53


In [17]:
# Crear un ranking de días según cantidad de viajes
consulta_ranking = """
SELECT
    pickup_day AS dia_semana,
    COUNT(*) AS cantidad_viajes,
    RANK() OVER (
        ORDER BY COUNT(*) DESC
    ) AS ranking_demanda
FROM trips
WHERE flag_quality_issue = FALSE
GROUP BY pickup_day
ORDER BY ranking_demanda
"""

con.execute(consulta_ranking).df()

,dia_semana,cantidad_viajes,ranking_demanda
0,Thursday,574632,1
1,Wednesday,569790,2
2,Friday,529043,3
3,Saturday,448676,4
4,Tuesday,428308,5
5,Sunday,416606,6
6,Monday,392564,7


In [18]:
# Combinar CTE y RANK para ordenar las horas según ingresos
consulta_cte_ranking = """
WITH ingresos_por_hora AS (
    SELECT
        pickup_hour AS hora,
        COUNT(*) AS cantidad_viajes,
        ROUND(SUM(total_amount), 2) AS ingresos_totales
    FROM trips_economic
    GROUP BY pickup_hour
)

SELECT
    hora,
    cantidad_viajes,
    ingresos_totales,
    RANK() OVER (
        ORDER BY ingresos_totales DESC
    ) AS ranking_ingresos
FROM ingresos_por_hora
ORDER BY ranking_ingresos
LIMIT 10
"""

con.execute(consulta_cte_ranking).df()

,hora,cantidad_viajes,ingresos_totales,ranking_ingresos
0,17,204631,"6,379,960.84",1
1,18,209892,"6,231,311.77",2
2,14,194652,"6,087,606.95",3
3,15,194450,"6,073,972.09",4
4,16,187156,"6,071,649.37",5
5,21,202686,"5,941,275.16",6
6,22,193070,"5,833,910.73",7
7,19,193621,"5,732,815.48",8
8,20,193657,"5,629,657.69",9
9,13,182659,"5,452,132.09",10


In [19]:
# Usar una subconsulta para comparar cada viaje con el importe promedio
consulta_subquery = """
SELECT
    COUNT(*) AS viajes_sobre_promedio
FROM trips_economic
WHERE total_amount > (
    SELECT AVG(total_amount)
    FROM trips_economic
)
"""

con.execute(consulta_subquery).df()

,viajes_sobre_promedio
0,1062860


In [20]:
# Clasificar los viajes según su importe total utilizando CASE
consulta_categoria_importe = """
SELECT
    CASE
        WHEN total_amount < 20 THEN 'Low'
        WHEN total_amount < 40 THEN 'Medium'
        WHEN total_amount < 70 THEN 'High'
        ELSE 'Very High'
    END AS categoria_importe,

    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(total_amount), 2) AS importe_promedio

FROM trips_economic

GROUP BY categoria_importe
ORDER BY importe_promedio
"""

con.execute(consulta_categoria_importe).df()

,categoria_importe,cantidad_viajes,importe_promedio
0,Low,1222468,15.67
1,Medium,1536134,27.59
2,High,372433,50.78
3,Very High,213545,94.99


In [21]:
# Contar las zonas de origen únicas con DISTINCT
consulta_distinct = """
SELECT
    COUNT(DISTINCT PULocationID) AS zonas_origen_unicas
FROM trips
WHERE flag_quality_issue = FALSE
"""

con.execute(consulta_distinct).df()

,zonas_origen_unicas
0,260


In [22]:
# Obtener valores mínimos y máximos con SQL
consulta_min_max = """
SELECT
    ROUND(MIN(total_amount), 2) AS importe_minimo,
    ROUND(MAX(total_amount), 2) AS importe_maximo,
    ROUND(MIN(trip_distance), 2) AS distancia_minima,
    ROUND(MAX(trip_distance), 2) AS distancia_maxima
FROM trips_economic
"""

con.execute(consulta_min_max).df()

,importe_minimo,importe_maximo,distancia_minima,distancia_maxima
0,1.00,720.84,0.01,99.60


In [23]:
# Obtener las 3 horas con mayor demanda de cada día
consulta_top3_horas = """
WITH demanda_hora_dia AS (
    SELECT
        pickup_day AS dia_semana,
        pickup_hour AS hora,
        COUNT(*) AS cantidad_viajes
    FROM trips
    WHERE flag_quality_issue = FALSE
    GROUP BY pickup_day, pickup_hour
),

ranking AS (
    SELECT
        dia_semana,
        hora,
        cantidad_viajes,
        ROW_NUMBER() OVER (
            PARTITION BY dia_semana
            ORDER BY cantidad_viajes DESC
        ) AS posicion
    FROM demanda_hora_dia
)

SELECT
    dia_semana,
    hora,
    cantidad_viajes,
    posicion
FROM ranking
WHERE posicion <= 3
ORDER BY dia_semana, posicion
"""

con.execute(consulta_top3_horas).df()

,dia_semana,hora,cantidad_viajes,posicion
0,Friday,23,32605,1
1,Friday,22,32432,2
2,Friday,18,31588,3
3,Monday,18,25953,1
4,Monday,20,25426,2
5,Monday,17,24577,3
6,Saturday,23,30034,1
7,Saturday,19,27975,2
8,Saturday,22,26543,3
9,Sunday,14,26698,1


In [24]:
# Calcular el cambio porcentual de demanda respecto a la hora anterior
consulta_cambio_pct = """
WITH demanda_hora AS (
    SELECT
        pickup_hour AS hora,
        COUNT(*) AS cantidad_viajes
    FROM trips
    WHERE flag_quality_issue = FALSE
    GROUP BY pickup_hour
),

comparacion AS (
    SELECT
        hora,
        cantidad_viajes,
        LAG(cantidad_viajes) OVER (
            ORDER BY hora
        ) AS viajes_hora_anterior
    FROM demanda_hora
)

SELECT
    hora,
    cantidad_viajes,
    viajes_hora_anterior,

    ROUND(
        (cantidad_viajes - viajes_hora_anterior)
        * 100.0 / viajes_hora_anterior,
        2
    ) AS cambio_pct

FROM comparacion
ORDER BY hora
"""

con.execute(consulta_cambio_pct).df()

,hora,cantidad_viajes,viajes_hora_anterior,cambio_pct
0,0,112193,<NA>,NaN
1,1,72893,112193,-35.03
2,2,48909,72893,-32.90
3,3,34809,48909,-28.83
4,4,29585,34809,-15.01
5,5,32455,29585,9.70
6,6,57641,32455,77.60
7,7,94106,57641,63.26
8,8,136065,94106,44.59
9,9,143673,136065,5.59


In [25]:
# Calcular la cantidad acumulada de viajes a lo largo del día
consulta_acumulado = """
WITH demanda_hora AS (
    SELECT
        pickup_hour AS hora,
        COUNT(*) AS cantidad_viajes
    FROM trips
    WHERE flag_quality_issue = FALSE
    GROUP BY pickup_hour
)

SELECT
    hora,
    cantidad_viajes,

    SUM(cantidad_viajes) OVER (
        ORDER BY hora
    ) AS viajes_acumulados

FROM demanda_hora
ORDER BY hora
"""

con.execute(consulta_acumulado).df()

,hora,cantidad_viajes,viajes_acumulados
0,0,112193,"112,193.00"
1,1,72893,"185,086.00"
2,2,48909,"233,995.00"
3,3,34809,"268,804.00"
4,4,29585,"298,389.00"
5,5,32455,"330,844.00"
6,6,57641,"388,485.00"
7,7,94106,"482,591.00"
8,8,136065,"618,656.00"
9,9,143673,"762,329.00"


In [26]:
# Calcular un promedio móvil de 3 horas
consulta_promedio_movil = """
WITH demanda_hora AS (
    SELECT
        pickup_hour AS hora,
        COUNT(*) AS cantidad_viajes
    FROM trips
    WHERE flag_quality_issue = FALSE
    GROUP BY pickup_hour
)

SELECT
    hora,
    cantidad_viajes,

    ROUND(
        AVG(cantidad_viajes) OVER (
            ORDER BY hora
            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        ),
        2
    ) AS promedio_movil_3h

FROM demanda_hora
ORDER BY hora
"""

con.execute(consulta_promedio_movil).df()

,hora,cantidad_viajes,promedio_movil_3h
0,0,112193,"92,543.00"
1,1,72893,"77,998.33"
2,2,48909,"52,203.67"
3,3,34809,"37,767.67"
4,4,29585,"32,283.00"
5,5,32455,"39,893.67"
6,6,57641,"61,400.67"
7,7,94106,"95,937.33"
8,8,136065,"124,614.67"
9,9,143673,"141,190.67"


In [27]:
# Analizar las combinaciones de zona y hora con mayores ingresos
consulta_zona_hora = """
SELECT
    z.Zone AS zona,
    t.pickup_hour AS hora,
    COUNT(*) AS cantidad_viajes,
    ROUND(SUM(t.total_amount), 2) AS ingresos_totales,
    ROUND(AVG(t.total_amount), 2) AS importe_promedio

FROM trips_economic AS t

LEFT JOIN read_csv_auto('../data/raw/taxi_zone_lookup.csv') AS z
    ON t.PULocationID = z.LocationID

GROUP BY
    z.Zone,
    t.pickup_hour

HAVING COUNT(*) >= 1000

ORDER BY ingresos_totales DESC

LIMIT 15
"""

con.execute(consulta_zona_hora).df()

,zona,hora,cantidad_viajes,ingresos_totales,importe_promedio
0,JFK Airport,16,10794,"949,013.21",87.92
1,JFK Airport,15,10141,"874,440.28",86.23
2,JFK Airport,17,9796,"839,650.25",85.71
3,JFK Airport,19,9276,"761,884.58",82.14
4,JFK Airport,14,8758,"745,712.83",85.15
5,JFK Airport,18,8737,"744,430.71",85.20
6,JFK Airport,20,9320,"710,298.76",76.21
7,JFK Airport,23,8782,"641,690.94",73.07
8,JFK Airport,22,8549,"625,550.35",73.17
9,JFK Airport,21,8131,"592,984.60",72.93


In [28]:
# Encontrar la hora con mayores ingresos dentro de cada zona
consulta_mejor_hora_zona = """
WITH ingresos_zona_hora AS (
    SELECT
        z.Zone AS zona,
        t.pickup_hour AS hora,
        COUNT(*) AS cantidad_viajes,
        SUM(t.total_amount) AS ingresos_totales

    FROM trips_economic AS t

    LEFT JOIN read_csv_auto('../data/raw/taxi_zone_lookup.csv') AS z
        ON t.PULocationID = z.LocationID

    GROUP BY
        z.Zone,
        t.pickup_hour
),

ranking AS (
    SELECT
        zona,
        hora,
        cantidad_viajes,
        ingresos_totales,

        RANK() OVER (
            PARTITION BY zona
            ORDER BY ingresos_totales DESC
        ) AS ranking

    FROM ingresos_zona_hora
)

SELECT
    zona,
    hora,
    cantidad_viajes,
    ROUND(ingresos_totales, 2) AS ingresos_totales

FROM ranking

WHERE ranking = 1

ORDER BY ingresos_totales DESC

LIMIT 15
"""

con.execute(consulta_mejor_hora_zona).df()

,zona,hora,cantidad_viajes,ingresos_totales
0,JFK Airport,16,10794,"949,013.21"
1,LaGuardia Airport,14,7307,"532,046.75"
2,Midtown Center,17,11648,"323,539.95"
3,Times Sq/Theatre District,22,9691,"280,235.00"
4,Midtown East,17,9014,"248,778.25"
5,Upper East Side South,17,10212,"239,000.43"
6,West Village,23,7733,"199,307.14"
7,Penn Station/Madison Sq West,22,6868,"191,984.36"
8,Upper East Side North,14,8556,"191,215.93"
9,Lincoln Square East,22,7560,"190,901.85"


In [29]:
# Crear una tabla resumen de KPIs por hora para Power BI
consulta_resumen_hora = """
SELECT
    pickup_hour AS hora,
    COUNT(*) AS cantidad_viajes,
    ROUND(SUM(total_amount), 2) AS ingresos_totales,
    ROUND(AVG(total_amount), 2) AS importe_promedio,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio,
    ROUND(AVG(trip_duration_min), 2) AS duracion_promedio

FROM trips_economic

GROUP BY pickup_hour

ORDER BY hora
"""

resumen_hora = con.execute(consulta_resumen_hora).df()

resumen_hora

,hora,cantidad_viajes,ingresos_totales,importe_promedio,distancia_promedio,duracion_promedio
0,0,111647,"3,527,753.66",31.60,4.52,16.40
1,1,72539,"2,153,860.59",29.69,4.41,15.41
2,2,48643,"1,392,034.94",28.62,4.33,15.02
3,3,34596,"1,045,923.88",30.23,4.75,15.30
4,4,29414,"987,748.66",33.58,5.45,15.70
5,5,32277,"1,092,866.22",33.86,5.58,15.22
6,6,57424,"1,713,240.79",29.83,4.64,14.82
7,7,93818,"2,703,545.25",28.82,3.98,15.55
8,8,135710,"3,924,637.74",28.92,3.47,16.97
9,9,143119,"4,107,759.43",28.70,3.29,17.13


In [30]:
# Exportar el resumen horario para Power BI
ruta_resumen_hora = "../data/processed/resumen_hora.csv"

resumen_hora.to_csv(
    ruta_resumen_hora,
    index=False
)

print("Archivo guardado correctamente:")
print(ruta_resumen_hora)

Archivo guardado correctamente:
../data/processed/resumen_hora.csv


In [31]:
# Crear una tabla resumen de KPIs por día de la semana
consulta_resumen_dia = """
SELECT
    pickup_day AS dia_semana,
    pickup_day_num AS numero_dia,
    COUNT(*) AS cantidad_viajes,
    ROUND(SUM(total_amount), 2) AS ingresos_totales,
    ROUND(AVG(total_amount), 2) AS importe_promedio,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio,
    ROUND(AVG(trip_duration_min), 2) AS duracion_promedio

FROM trips_economic

GROUP BY
    pickup_day,
    pickup_day_num

ORDER BY numero_dia
"""

resumen_dia = con.execute(consulta_resumen_dia).df()

resumen_dia

,dia_semana,numero_dia,cantidad_viajes,ingresos_totales,importe_promedio,distancia_promedio,duracion_promedio
0,Monday,0,390823,"12,092,850.14",30.94,3.75,16.84
1,Tuesday,1,426425,"12,663,329.79",29.70,3.32,17.08
2,Wednesday,2,567263,"17,030,066.73",30.02,3.34,17.36
3,Thursday,3,572090,"17,325,487.18",30.28,3.37,17.97
4,Friday,4,526439,"15,783,379.30",29.98,3.55,17.57
5,Saturday,5,446662,"13,030,218.06",29.17,3.67,16.85
6,Sunday,6,414878,"12,808,492.62",30.87,4.17,16.67


In [32]:
# Exportar el resumen por día de la semana para Power BI
ruta_resumen_dia = "../data/processed/resumen_dia_semana.csv"

resumen_dia.to_csv(
    ruta_resumen_dia,
    index=False
)

print("Archivo guardado correctamente:")
print(ruta_resumen_dia)

Archivo guardado correctamente:
../data/processed/resumen_dia_semana.csv


In [33]:
# Crear una tabla resumen de KPIs por zona de origen
consulta_resumen_zona = """
SELECT
    t.PULocationID AS location_id,
    z.Borough AS borough,
    z.Zone AS zona,

    COUNT(*) AS cantidad_viajes,

    ROUND(SUM(t.total_amount), 2) AS ingresos_totales,

    ROUND(AVG(t.total_amount), 2) AS importe_promedio,

    ROUND(AVG(t.trip_distance), 2) AS distancia_promedio,

    ROUND(AVG(t.trip_duration_min), 2) AS duracion_promedio

FROM trips_economic AS t

LEFT JOIN read_csv_auto('../data/raw/taxi_zone_lookup.csv') AS z
    ON t.PULocationID = z.LocationID

GROUP BY
    t.PULocationID,
    z.Borough,
    z.Zone

ORDER BY ingresos_totales DESC
"""

resumen_zona = con.execute(consulta_resumen_zona).df()

resumen_zona.head(15)

,location_id,borough,zona,cantidad_viajes,ingresos_totales,importe_promedio,distancia_promedio,duracion_promedio
0,132,Queens,JFK Airport,145457,"11,430,997.60",78.59,14.77,39.37
1,138,Queens,LaGuardia Airport,86100,"5,998,248.68",69.67,9.63,31.04
2,161,Manhattan,Midtown Center,146618,"3,959,926.59",27.01,2.38,15.87
3,230,Manhattan,Times Sq/Theatre District,104249,"3,176,546.85",30.47,3.07,17.66
4,186,Manhattan,Penn Station/Madison Sq West,113244,"3,058,805.24",27.01,2.44,17.25
5,237,Manhattan,Upper East Side South,134357,"2,932,226.65",21.82,1.86,12.42
6,162,Manhattan,Midtown East,108544,"2,813,969.26",25.92,2.33,15.16
7,68,Manhattan,East Chelsea,91537,"2,549,386.73",27.85,2.72,17.13
8,236,Manhattan,Upper East Side North,112556,"2,541,740.85",22.58,2.20,12.76
9,170,Manhattan,Murray Hill,92609,"2,358,957.91",25.47,2.38,15.08


In [34]:
# Exportar el resumen por zona de origen para Power BI
ruta_resumen_zona = "../data/processed/resumen_zona_origen.csv"

resumen_zona.to_csv(
    ruta_resumen_zona,
    index=False
)

print("Archivo guardado correctamente:")
print(ruta_resumen_zona)

Archivo guardado correctamente:
../data/processed/resumen_zona_origen.csv


In [35]:
# Crear una tabla con los principales KPIs del proyecto
consulta_kpis_powerbi = """
SELECT
    COUNT(*) AS total_viajes,

    ROUND(SUM(total_amount), 2) AS ingresos_totales,

    ROUND(AVG(total_amount), 2) AS importe_promedio,

    ROUND(MEDIAN(total_amount), 2) AS importe_mediano,

    ROUND(AVG(trip_distance), 2) AS distancia_promedio,

    ROUND(AVG(trip_duration_min), 2) AS duracion_promedio

FROM trips_economic
"""

kpis_powerbi = con.execute(consulta_kpis_powerbi).df()

kpis_powerbi

,total_viajes,ingresos_totales,importe_promedio,importe_mediano,distancia_promedio,duracion_promedio
0,3344580,"100,733,823.82",30.12,23.57,3.57,17.25


In [36]:
# Exportar los KPIs generales para Power BI
ruta_kpis = "../data/processed/kpis_generales.csv"

kpis_powerbi.to_csv(
    ruta_kpis,
    index=False
)

print("Archivo guardado correctamente:")
print(ruta_kpis)

Archivo guardado correctamente:
../data/processed/kpis_generales.csv


## Conclusiones del análisis SQL

El análisis SQL permitió consultar y resumir los datos procesados de Yellow Taxi de julio de 2026 utilizando DuckDB directamente sobre archivos Parquet.

### Principales resultados

- Se identificaron **3.344.580 viajes válidos para el análisis económico**.
- Los ingresos totales analizados alcanzaron aproximadamente **100,73 millones USD**.
- El importe promedio por viaje fue de **30,12 USD**, mientras que la mediana fue de **23,57 USD**.
- La distancia promedio fue de **3,57 millas**.
- La duración promedio de los viajes fue de **17,25 minutos**.
- Las **18:00** registraron la mayor cantidad de viajes, mientras que las **17:00** generaron los mayores ingresos.
- El jueves presentó el mayor volumen de viajes y de ingresos entre los días de la semana.
- JFK Airport destacó por su elevada generación de ingresos en comparación con otras zonas de origen.

### Técnicas SQL utilizadas

Durante el análisis se utilizaron:

- filtros con `WHERE`;
- agregaciones con `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` y `MEDIAN`;
- agrupaciones con `GROUP BY`;
- filtros sobre agregaciones con `HAVING`;
- `CASE` para clasificación de datos;
- `JOIN` para incorporar información geográfica;
- subconsultas y CTE;
- funciones de ventana como `RANK`, `ROW_NUMBER` y `LAG`;
- cálculos acumulados y promedios móviles con funciones `OVER`;
- cuartiles mediante `QUANTILE_CONT`.

### Tablas generadas para Power BI

Se exportaron cuatro tablas agregadas:

- `kpis_generales.csv`
- `resumen_hora.csv`
- `resumen_dia_semana.csv`
- `resumen_zona_origen.csv`

Estas tablas permiten continuar el proyecto en Power BI utilizando datasets agregados y optimizados para visualización.